# 📦 第3课：函数 — 把代码打包复用

---

### 🎯 学完这课你能：
- 把 "计算收益率"、"生成信号" 封装成可复用的函数
- 写出计算最大回撤的函数
- 用函数组合一个简单的策略回测

### 📌 为什么需要函数？
```
没有函数 → 相同代码复制粘贴 100 次 😵
有函数   → 写一次，调用 100 次 😎
```

## 1️⃣ 定义函数 — `def`

In [ ]:
def calculate_profit(buy_price, sell_price, shares):
    """计算交易盈亏和收益率"""
    profit = (sell_price - buy_price) * shares
    rate = (sell_price - buy_price) / buy_price * 100
    return profit, rate  # 返回多个值

# 调用函数 — 一行搞定
p1, r1 = calculate_profit(150, 178.52, 100)
p2, r2 = calculate_profit(60000, 65000, 2)

print(f"AAPL: 盈亏 ${p1:,.2f}  收益率 {r1:.2f}%")
print(f"BTC:  盈亏 ${p2:,.2f}  收益率 {r2:.2f}%")

## 2️⃣ 默认参数

In [ ]:
def sma(prices, window=20):
    """简单移动平均线"""
    if len(prices) < window:
        return None
    return sum(prices[-window:]) / window

prices = [100, 102, 101, 105, 103, 107, 106, 110, 108, 112]

print(f"SMA(5)  = {sma(prices, 5):.2f}")    # 指定 window=5
print(f"SMA(3)  = {sma(prices, 3):.2f}")    # 指定 window=3
print(f"SMA(20) = {sma(prices)}")            # 默认 20，数据不够返回 None

## 3️⃣ 实战：双均线交叉信号

In [ ]:
def ma_signal(prices, short=5, long=10):
    """双均线交叉策略信号
    
    金叉（短均线 > 长均线）→ 买入
    死叉（短均线 < 长均线）→ 卖出
    """
    if len(prices) < long:
        return "⚠️ 数据不足"
    
    short_ma = sum(prices[-short:]) / short
    long_ma = sum(prices[-long:]) / long
    diff = short_ma - long_ma
    
    print(f"  MA({short}) = {short_ma:.2f}")
    print(f"  MA({long}) = {long_ma:.2f}")
    print(f"  差值 = {diff:+.2f}")
    
    if diff > 0:
        return "🟢 金叉 → 买入信号"
    else:
        return "🔴 死叉 → 卖出信号"

# 上涨趋势
up_prices = [100, 101, 99, 102, 104, 103, 106, 108, 107, 110, 112, 115]
print("📈 上涨趋势:")
print(f"  → {ma_signal(up_prices)}")

print()

# 下跌趋势
down_prices = [115, 112, 113, 110, 108, 109, 106, 104, 105, 102, 100, 98]
print("📉 下跌趋势:")
print(f"  → {ma_signal(down_prices)}")

## 4️⃣ 实战：计算最大回撤

**最大回撤 (Max Drawdown)** — 你投入后最惨能亏多少

$$MDD = \frac{谷底 - 峰值}{峰值} \times 100\%$$

这是评估策略风险最重要的指标之一。

In [ ]:
def max_drawdown(prices):
    """计算最大回撤"""
    peak = prices[0]     # 历史最高点
    mdd = 0              # 最大回撤
    peak_idx = 0
    valley_idx = 0
    
    for i, price in enumerate(prices):
        if price > peak:
            peak = price
            peak_idx = i
        
        dd = (price - peak) / peak * 100
        if dd < mdd:
            mdd = dd
            valley_idx = i
    
    return mdd, peak_idx, valley_idx

# 测试
test = [100, 110, 120, 105, 115, 90, 95, 100, 130, 125]
mdd, pi, vi = max_drawdown(test)

print("价格走势可视化：")
for i, p in enumerate(test):
    bar = "█" * int(p / 3)
    marker = ""
    if i == pi: marker = " ← 📍 峰值"
    if i == vi: marker = " ← 📍 谷底"
    print(f"  {p:>4} {bar}{marker}")

print(f"\n最大回撤: {mdd:.2f}%")
print(f"从 {test[pi]} 跌到 {test[vi]}")

## 5️⃣ 函数组合：极简策略回测

In [ ]:
import random
random.seed(123)

def simple_backtest(prices, buy_dip=-0.02, sell_up=0.03):
    """极简回测：跌 X% 买入，累计涨 Y% 卖出"""
    holding = False
    buy_price = 0
    trades = []
    
    for i in range(1, len(prices)):
        daily_ret = (prices[i] - prices[i-1]) / prices[i-1]
        
        if not holding and daily_ret < buy_dip:
            buy_price = prices[i]
            holding = True
            print(f"  Day{i:>3} 🟢 买入 @ ${buy_price:.2f}")
        
        elif holding:
            pnl = (prices[i] - buy_price) / buy_price
            if pnl > sell_up:
                holding = False
                trades.append(pnl * 100)
                print(f"  Day{i:>3} 🔴 卖出 @ ${prices[i]:.2f}  收益: {pnl*100:+.2f}%")
    
    if not trades:
        print("  没有交易产生")
        return
    
    wins = [t for t in trades if t > 0]
    losses = [t for t in trades if t <= 0]
    
    print(f"\n📊 回测结果")
    print(f"  ├─ 交易次数: {len(trades)}")
    print(f"  ├─ 胜率: {len(wins)/len(trades)*100:.0f}%")
    print(f"  ├─ 平均收益: {sum(trades)/len(trades):.2f}%")
    print(f"  └─ 总收益: {sum(trades):.2f}%")

# 生成 30 天模拟数据
sim = [100]
for _ in range(50):
    sim.append(sim[-1] * (1 + random.uniform(-0.04, 0.04)))

simple_backtest(sim)

---

## 🏋️ 动手练习

1. 修改 `simple_backtest` 的 `buy_dip` 和 `sell_up` 参数，观察结果变化
2. 写一个函数 `daily_returns(prices)` 返回每日收益率列表
3. 写一个函数 `sharpe_ratio(returns)` 计算夏普比率：`平均收益率 / 标准差`

In [ ]:
# 👇 在这里练习



---
## ✅ 第3课完成！

**你学会了：** 定义函数 / 默认参数 / 多返回值 / 函数组合做回测

**下一课** → `04_数据结构.ipynb`